<h1>Table of Contents<span class="tocSkip"></span></h1>
<div class="toc"><ul class="toc-item"><li><span><a href="#Loading-data" data-toc-modified-id="Loading-data-1">Loading data</a></span></li><li><span><a href="#Quick-Checks" data-toc-modified-id="Quick-Checks-2">Quick Checks</a></span></li><li><span><a href="#Create-Allelic-requirement-mapping" data-toc-modified-id="Create-Allelic-requirement-mapping-3">Create Allelic requirement mapping</a></span></li><li><span><a href="#Create-molecular_mechanism-mapping" data-toc-modified-id="Create-molecular_mechanism-mapping-4">Create molecular_mechanism mapping</a></span></li><li><span><a href="#Filter" data-toc-modified-id="Filter-5">Filter</a></span><ul class="toc-item"><li><span><a href="#Confidence" data-toc-modified-id="Confidence-5.1">Confidence</a></span></li><li><span><a href="#disease-IDs-(missing,-etc)" data-toc-modified-id="disease-IDs-(missing,-etc)-5.2">disease IDs (missing, etc)</a></span></li></ul></li><li><span><a href="#Simple-processing" data-toc-modified-id="Simple-processing-6">Simple processing</a></span><ul class="toc-item"><li><span><a href="#publications" data-toc-modified-id="publications-6.1">publications</a></span></li><li><span><a href="#source-record-url" data-toc-modified-id="source-record-url-6.2">source record url</a></span></li><li><span><a href="#date-(last-reviewed)" data-toc-modified-id="date-(last-reviewed)-6.3">date (last reviewed)</a></span></li><li><span><a href="#ID-prefixes" data-toc-modified-id="ID-prefixes-6.4">ID prefixes</a></span></li></ul></li><li><span><a href="#Comparing-to-parser-output" data-toc-modified-id="Comparing-to-parser-output-7">Comparing to parser output</a></span></li><li><span><a href="#OLD:-Pre-NodeNorming" data-toc-modified-id="OLD:-Pre-NodeNorming-8">OLD: Pre-NodeNorming</a></span><ul class="toc-item"><li><span><a href="#Exploring:-Genes" data-toc-modified-id="Exploring:-Genes-8.1">Exploring: Genes</a></span><ul class="toc-item"><li><span><a href="#HGNC" data-toc-modified-id="HGNC-8.1.1">HGNC</a></span></li><li><span><a href="#OMIM" data-toc-modified-id="OMIM-8.1.2">OMIM</a></span></li><li><span><a href="#Comparing-HGNC-vs-OMIM" data-toc-modified-id="Comparing-HGNC-vs-OMIM-8.1.3">Comparing HGNC vs OMIM</a></span></li><li><span><a href="#Conclusions" data-toc-modified-id="Conclusions-8.1.4">Conclusions</a></span></li></ul></li><li><span><a href="#Exploring:-Diseases" data-toc-modified-id="Exploring:-Diseases-8.2">Exploring: Diseases</a></span><ul class="toc-item"><li><span><a href="#OMIM/orphanet" data-toc-modified-id="OMIM/orphanet-8.2.1">OMIM/orphanet</a></span></li><li><span><a href="#MONDO" data-toc-modified-id="MONDO-8.2.2">MONDO</a></span></li><li><span><a href="#Comparing-OMIM/orphanet-vs-MONDO" data-toc-modified-id="Comparing-OMIM/orphanet-vs-MONDO-8.2.3">Comparing OMIM/orphanet vs MONDO</a></span></li><li><span><a href="#Checking-MONDO-data" data-toc-modified-id="Checking-MONDO-data-8.2.4">Checking MONDO data</a></span></li><li><span><a href="#Conclusions" data-toc-modified-id="Conclusions-8.2.5">Conclusions</a></span></li></ul></li><li><span><a href="#Stats-on-rows-removed-during-NodeNorming" data-toc-modified-id="Stats-on-rows-removed-during-NodeNorming-8.3">Stats on rows removed during NodeNorming</a></span></li><li><span><a href="#Adding-NodeNorm-data,-removing-rows" data-toc-modified-id="Adding-NodeNorm-data,-removing-rows-8.4">Adding NodeNorm data, removing rows</a></span></li></ul></li><li><span><a href="#OLD:-Generating-documents" data-toc-modified-id="OLD:-Generating-documents-9">OLD: Generating documents</a></span><ul class="toc-item"><li><span><a href="#BioThings-type-parser" data-toc-modified-id="BioThings-type-parser-9.1">BioThings-type parser</a></span></li><li><span><a href="#File:-List-of-TRAPI-edges" data-toc-modified-id="File:-List-of-TRAPI-edges-9.2">File: List of TRAPI edges</a></span></li><li><span><a href="#File:-KGX-edges" data-toc-modified-id="File:-KGX-edges-9.3">File: KGX edges</a></span></li><li><span><a href="#File:-KGX-nodes" data-toc-modified-id="File:-KGX-nodes-9.4">File: KGX nodes</a></span></li></ul></li><li><span><a href="#DEFUNCT:-Checking-documents" data-toc-modified-id="DEFUNCT:-Checking-documents-10">DEFUNCT: Checking documents</a></span></li><li><span><a href="#BioThings-Parser-notes" data-toc-modified-id="BioThings-Parser-notes-11">BioThings Parser notes</a></span></li></ul></div>

# Notebook for EBI gene2pheno parser development

In [1]:
## not for parser. for notebook only 

## CX: allows multiple lines of code to print from one code block
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

## Loading data

In [2]:
## INCLUDE IN PARSER
import pandas as pd
import requests
from datetime import datetime

## unsure on putting into parser: more for notebook viewing/debugging...
pd.options.display.max_columns = None
# pd.set_option('display.max_colwidth', None)
# pd.reset_option('display.max_colwidth')

In [3]:
## useful function for more EDA
def check_if_contains(df, column_name, patterns):
    for i in patterns:
        temp = df[df[column_name].str.contains(pat=i, na=False)]
        if temp.size > 0:
            print(f'"{i}"')
            print(temp.shape)

Using the "on-the-fly", "all data" download thru the API. I think this is the easiest to set up the common ingest pipeline to use. It has all data **without duplicates** (checked in data-playground notebook).

In [4]:
## to mimic PARSER

## get current time of download for common ingest pipeline
## from https://stackoverflow.com/a/69080715 
datetime.now(datetime.now().astimezone().tzinfo).strftime("%Y-%m-%d")

'2026-03-03'

In [5]:
## INCLUDE IN PARSER, adjusted

all_data_url = "https://www.ebi.ac.uk/gene2phenotype/api/panel/all/download/"

## from data-playground: force some columns to ingest as str
df = pd.read_csv(all_data_url, 
                 dtype={
                     "gene mim": str,
                     "hgnc id": str,
                     "disease mim": str,
                 }
                )

## make column names snake-case - usable with itertuples later
# df.columns = df.columns.str.replace(" ", "_")

In [6]:
df.info(memory_usage="deep")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3826 entries, 0 to 3825
Data columns (total 24 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   g2p id                              3826 non-null   object 
 1   gene symbol                         3826 non-null   object 
 2   gene mim                            3823 non-null   object 
 3   hgnc id                             3826 non-null   object 
 4   previous gene symbols               3529 non-null   object 
 5   disease name                        3826 non-null   object 
 6   disease mim                         3031 non-null   object 
 7   disease MONDO                       3413 non-null   object 
 8   allelic requirement                 3826 non-null   object 
 9   cross cutting modifier              477 non-null    object 
 10  confidence                          3826 non-null   object 
 11  variant consequence                 3807 no

In [7]:
df.head()

,g2p id,gene symbol,gene mim,hgnc id,previous gene symbols,disease name,disease mim,disease MONDO,allelic requirement,cross cutting modifier,confidence,variant consequence,variant types,molecular mechanism,molecular mechanism support,molecular mechanism categorisation,molecular mechanism evidence,phenotypes,publications,additional mined publications,panel,comments,date of last review,review
0,G2P00001,HMX1,142992,5017,H6; NKX5-3,HMX1-related oculoauricular syndrome,612109,MONDO:0012802,biallelic_autosomal,NaN,definitive,absent gene product,NaN,loss of function,inferred,NaN,NaN,HP:0000007; HP:0000356; HP:0000482; HP:0000505...,18423520; 25574057; 29140751,21417677; 31469246; 35946463,DD; Eye,NaN,2019-09-26 16:23:46+00:00,NaN
1,G2P00002,SLX4,613278,23845,BTBD12; FANCP; KIAA1784; KIAA1987,SLX4-related Fanconi anemia,613951,MONDO:0013499,biallelic_autosomal,NaN,definitive,absent gene product,NaN,loss of function,inferred,NaN,NaN,HP:0000007; HP:0000028; HP:0000085; HP:0000125...,21240275; 21240277,21476996; 23033263; 26841305; 30047418; 347541...,DD,NaN,2025-01-28 23:09:54+00:00,NaN
2,G2P00003,ARG1,608313,663,NaN,ARG1-related argininemia,207800,MONDO:0008814,biallelic_autosomal,NaN,definitive,absent gene product,NaN,loss of function,inferred,NaN,NaN,HP:0000007; HP:0000737; HP:0000752; HP:0001249...,10502833; 1463019; 1598908; 2365823; 7649538,15565656; 19052914; 19562505; 19936428; 213103...,DD,NaN,2015-07-22 16:14:07+00:00,NaN
3,G2P00004,ATR,601215,882,FRP1; MEC1; SCKL; SCKL1,ATR-related Seckel syndrome,210600,MONDO:0008869,biallelic_autosomal,NaN,strong,absent gene product,NaN,loss of function,inferred,NaN,NaN,HP:0000007; HP:0000028; HP:0000047; HP:0000175...,10889046; 12640452; 23111928; 30199583,10631133; 15279811; 21669506; 23144622; 272354...,DD; Skeletal,NaN,2025-01-27 14:24:27+00:00,NaN
4,G2P00005,FANCB,300515,3583,FAAP95; FAB; FLJ34064,FANCB-related Fanconi anemia,300514,MONDO:0010351,monoallelic_X_hemizygous,NaN,definitive,absent gene product,NaN,loss of function,inferred,NaN,NaN,HP:0000083; HP:0000100; HP:0000119; HP:0000707...,15502827; 16679491; 21910217; 22052692; 236135...,24416387; 31351673,DD; Skin,NaN,2024-08-20 14:13:58+00:00,NaN


## Quick Checks

"disease mim" column used to have both omim and orphanet IDs

In [10]:
## include in parser as log/debug


df[df["disease mim"].isna() & df["disease MONDO"].isna()].shape[0]

df["publications"].isna().sum()

223

np.int64(99)

In [11]:
## include in parser as log

df[df["disease mim"].str.contains("orpha", case=False, na=False)]

df["disease mim"].str.contains("orpha", case=False, na=False).sum()

print(f"{df["disease mim"].str.contains("orpha", case=False, na=False).sum()} rows with orphanet ID in 'disease mim' column")

,g2p id,gene symbol,gene mim,hgnc id,previous gene symbols,disease name,disease mim,disease MONDO,allelic requirement,cross cutting modifier,confidence,variant consequence,variant types,molecular mechanism,molecular mechanism support,molecular mechanism categorisation,molecular mechanism evidence,phenotypes,publications,additional mined publications,panel,comments,date of last review,review


np.int64(0)

0 rows with orphanet ID in 'disease mim' column


In [12]:
## include in parser as log

df[df.duplicated(keep=False)]

df.duplicated(keep=False).sum()

print(f"{df.duplicated(keep=False).sum()} duplicate rows")

,g2p id,gene symbol,gene mim,hgnc id,previous gene symbols,disease name,disease mim,disease MONDO,allelic requirement,cross cutting modifier,confidence,variant consequence,variant types,molecular mechanism,molecular mechanism support,molecular mechanism categorisation,molecular mechanism evidence,phenotypes,publications,additional mined publications,panel,comments,date of last review,review


np.int64(0)

0 duplicate rows


## Create Allelic requirement mapping

"Allelic requirement" has been [added to biolink-model](https://github.com/biolink/biolink-model/pull/1576) as an edge property/attribute. 

* **The values are supposed to be HP terms**, so we need to map [EBI Gene2Phenotype's allelic_requirement values](https://www.ebi.ac.uk/gene2phenotype/about/terminology#allelic-requirement-section) to their corresponding HP terms
* **I'm going to map all possible values**, even if they don't show up in the data right now. 
* using the OLS API (downloads the latest HPO release). 
  * this OLS endpoint can't query on or show the type of synonym (ref: [github issue](https://github.com/EBISPOT/ols4/issues/917)). The API docs are [here](https://www.ebi.ac.uk/ols4/help) (they're long and confusing). 
  * I'm not using the [HP API](https://ontology.jax.org/api/hp/docs) because I found it unreliable: have to replace(`"_"`, `" "`) allelic_requirement values to search, then search results include many incorrect other IDs.

In [13]:
## actual values in data
df["allelic requirement"].value_counts()

allelic requirement
biallelic_autosomal           2002
monoallelic_autosomal         1582
monoallelic_X_hemizygous       174
monoallelic_X_heterozygous      58
mitochondrial                    7
monoallelic_X                    2
monoallelic_Y_hemizygous         1
Name: count, dtype: int64

In [14]:
## INCLUDE IN PARSER

## only the values for dynamic mappings - HP has correct synonyms

ALLELIC_REQ_TO_MAP = [
    "biallelic_autosomal",
    "monoallelic_autosomal",
    "biallelic_PAR",
    "monoallelic_PAR",
    "mitochondrial",
    "monoallelic_Y_hemizygous",
    "monoallelic_X",
    "monoallelic_X_hemizygous",
    "monoallelic_X_heterozygous",
]

In [15]:
## INCLUDE IN PARSER

## requires requests library

def build_allelic_req_mappings(allelic_req_val):
    ## queries OLS to find what HP term has the allelic requirement value as an exact synonym (OLS uses the latest HPO release)
    ols_request = (
        f"https://www.ebi.ac.uk/ols4/api/search?q={allelic_req_val}&ontology=hp&queryFields=synonym&exact=true"
    )
    try:
        response = requests.get(ols_request, timeout=5)
        if response.status_code == 200:
            temp = response.json()
            return temp["response"]["docs"][0]["obo_id"]  ## only need the HP ID
        else:
            print(f"Error encountered on '{allelic_req_val}': {response.status_code}")
    except requests.RequestException as e:
        print(f"Request exemption encountered on '{allelic_req_val}': {e}")

In [16]:
## INCLUDE IN PARSER

## using dictionary comprehension
allelicreq_mappings = {i: build_allelic_req_mappings(i) for i in ALLELIC_REQ_TO_MAP}

allelicreq_mappings

{'biallelic_autosomal': 'HP:0000007',
 'monoallelic_autosomal': 'HP:0000006',
 'biallelic_PAR': 'HP:0034341',
 'monoallelic_PAR': 'HP:0034340',
 'mitochondrial': 'HP:0001427',
 'monoallelic_Y_hemizygous': 'HP:0001450',
 'monoallelic_X': 'HP:0001417',
 'monoallelic_X_hemizygous': 'HP:0001419',
 'monoallelic_X_heterozygous': 'HP:0001423'}

<div class="alert alert-block alert-danger">

Check for accuracy against my manually-reviewed mappings (from data-playground notebook)
    
| G2P | HPO name | HPO ID | Exact synonym? | 
| :- | :- | :- | :- |
| biallelic_autosomal | Autosomal recessive inheritance | HP:0000007 | Yes |
| monoallelic_autosomal | Autosomal dominant inheritance | HP:0000006 | Yes |
| biallelic_PAR | Pseudoautosomal recessive inheritance | HP:0034341 | Yes |
| monoallelic_PAR | Pseudoautosomal dominant inheritance | HP:0034340 | Yes |
| mitochondrial | Mitochondrial inheritance | HP:0001427 | Yes |
| monoallelic_Y_hemizygous | Y-linked inheritance | HP:0001450 | Yes |
| **monoallelic_X** | X-linked inheritance | HP:0001417 | Yes |
| **monoallelic_X_hemizygous** | X-linked recessive inheritance | HP:0001419 | Yes |
| **monoallelic_X_heterozygous** | X-linked dominant inheritance | HP:0001423 | Yes |

## Create molecular_mechanism mapping

The molecular_mechanism [terms](https://www.ebi.ac.uk/gene2phenotype/about/terminology#molecular-mechanism-section) describe the genotype (of gene variants) linked to the disease. 

**FOR NOW**, mapping to biolink `subject_form_or_variant_qualifier` (Gene). Values are `genetic_variant_form` or a descendant. I manually create these mappings using the definitions from the EBI Gene2Pheno terminology page - it's straightforward.

Related biolink-model change: https://github.com/biolink/biolink-model/pull/1575

In [17]:
## code chunk to review data

df["molecular mechanism"].value_counts()

molecular mechanism
loss of function                     2274
undetermined                         1220
gain of function                      194
dominant negative                     130
undetermined non-loss-of-function       8
Name: count, dtype: int64

In [18]:
## INCLUDE IN PARSER

FORM_OR_VARIANT_QUALIFIER_MAPPINGS = {
    "loss of function": "loss_of_function_variant_form",
    "undetermined": "genetic_variant_form",
    "gain of function": "gain_of_function_variant_form",
    "dominant negative": "dominant_negative_variant_form",
    "undetermined non-loss-of-function": "non_loss_of_function_variant_form"
}

## Filter

### Confidence

**2025-07-22:**

Every row/record has 1 confidence value, representing how confident the curators are that "this gene has a causal role in this disease". The definitions of the possible values are provided [here](https://www.ebi.ac.uk/gene2phenotype/about/terminology#g2p-confidence-section). 


**CURRENT DEFINITIONS** (including in case they change later)

> **definitive**: The role of this gene in this particular disease has been repeatedly demonstrated in both the research and clinical diagnostic settings, and has been upheld over time (at least 2 independent publication over 3 years' time). No convincing evidence has emerged that contradicts the role of the gene in the specified disease. (previously labelled as confirmed) The strength of evidence within publications as well as their number and publication dates is taken into account. In practice, this usually means at least 4 publications over 5 years. Typically this will also include convincing bioinformatic or functional evidence of causation, making it very unlikely that this gene-disease association would ever be refuted.
>
>**strong**: The role of this gene as a monogenic cause of disease has been repeatedly and independently demonstrated providing very strong convincing evidence in humans and no conflicting evidence for this gene's role in this disease. (previously labelled as probable).
>
>**moderate**: There is moderate evidence in humans to support a casual role for this gene in this disease with no contradictory evidence. The body of evidence is not large (e.g possibly only one key paper) but appears convincing enough that the gene-disease pair is likely to be validated with additional evidence in the near future.
>
>**limited**: Little human evidence exists to support a casual role for this gene in this disease, but not all evidence has been refuted. For example, there may be a collection of rare missense variants in humans but without convincing functional impact, segregration data that could either arise by chance (e.g across one or two meioses) or does not implicate a single gene, or functional data without direct recapitulation of the phenotype. Overall, the body of evidence does not meet contemporary criteria for claiming a valid association with disease. The majority are probably false associations. (previously labelled as possible).
>
>**disputed**: Although evidence has been reported, other evidence of equal weight disputes the claim.
>
>**refuted**: There has been an assertion of a gene-disease association in the literature, but new valid evidence has arisen that refutes the entire original body of evidence.

<div class="alert alert-block alert-success">

**2025-07-22:**

After [input from Sierra and Matt](https://github.com/biolink/biolink-model/issues/1581#issuecomment-3120242153):
1. **FOR NOW**, rows with **"refuted"** or **"disputed"** values **should not be used** to create edges for Translator. These mean there's strong evidence that there ISN'T an association (negation). **This decision can be revisited** once Translator can model/handle negation better. 
2. rows with **"limited"** value **should not be used** to create edges for Translator. Sierra and Matt pointed out the last sentence of the definition: "The majority are probably false associations. (previously labeled as possible)." The reasoning is that these may not be "real" associations. 
3. keep rows with **"moderate", "strong", "definitive"** values, because there's moderate-definitive evidence that a gene DOES HAVE a causal role in this disease. 

<div class="alert alert-block alert-danger">

Data-modeling notes: predicate options for gene-disease associations are confusing 
* "causes / contributes to" makes more sense when qualifiers on the gene/protein are used. 
* what's the diff between "associated with" and "genetically associated with"? 
* "gene associated with condition" is child of "genetically associated with", but seems to be more general - basically a "related to". Also would look weird in UI, right? 

In [19]:
df["confidence"].value_counts()

confidence
definitive    2038
strong         858
limited        582
moderate       295
disputed        50
refuted          3
Name: count, dtype: int64

In [20]:
## INCLUDE IN PARSER 

CONFIDENCE_TO_FILTER = ["limited", "disputed", "refuted"]

In [21]:
## INCLUDE IN PARSER 

df = df[~ df["confidence"].isin(CONFIDENCE_TO_FILTER)]

In [22]:
## INCLUDE IN PARSER - log

print(f"{df.shape[0]} rows after removing confidence values: {", ".join(CONFIDENCE_TO_FILTER)}")

3191 rows after removing confidence values: limited, disputed, refuted


### disease IDs (missing, etc)

Not doing on gene IDs because the column `hgnc id` consistently doesn't have any missing values. 

In [23]:
## INCLUDE IN PARSER 
## drop if no disease ID

df.dropna(how="all", subset=["disease mim", "disease MONDO"], inplace=True, ignore_index=True)

print(f"{df.shape[0]} rows after removing rows with no disease ID")

3062 rows after removing rows with no disease ID


<div class="alert alert-block alert-danger">

**TEMP FILTER** on [OMIM:188400](https://omim.org/entry/188400)
    
NodeNorm incorrectly assigns this to a Gene when it's actually a Disease, causing an incorrect edge to be made
    
Can undo this filter once NodeNorm is fixed. 

In [24]:
## EDA

df[(df["disease mim"] == "188400")]

,g2p id,gene symbol,gene mim,hgnc id,previous gene symbols,disease name,disease mim,disease MONDO,allelic requirement,cross cutting modifier,confidence,variant consequence,variant types,molecular mechanism,molecular mechanism support,molecular mechanism categorisation,molecular mechanism evidence,phenotypes,publications,additional mined publications,panel,comments,date of last review,review
136,G2P00152,TBX1,602054,11592,CATCH22; VCF,TBX1-related 22q11.2 deletion syndrome,188400,MONDO:0008564,monoallelic_autosomal,NaN,definitive,absent gene product,NaN,loss of function,inferred,NaN,NaN,HP:0000006; HP:0000023; HP:0000110; HP:0000122...,14585638,10440825; 11748311; 12668595; 12891520; 131297...,DD,NaN,2015-07-22 16:14:19+00:00,NaN


In [25]:
## INCLUDE IN PARSER
## temp filter out 

TEMP_OMIM_FILTER = ["188400"]

df = df[~ df["disease mim"].isin(TEMP_OMIM_FILTER)]

In [26]:
## INCLUDE IN PARSER - log

print(f"{df.shape[0]} rows after removing disease OMIM: {", ".join(TEMP_OMIM_FILTER)}")

3061 rows after removing disease OMIM: 188400


## Simple processing

### publications

In [31]:
df["publications"].isna().sum()

df["publications"][0:3]

np.int64(80)

0                    18423520; 25574057; 29140751
1                              21240275; 21240277
2    10502833; 1463019; 1598908; 2365823; 7649538
Name: publications, dtype: object

In [29]:
## in parser, adjusted for single-row-level transform

processed_publications = []
for x in df["publications"]: 
    if pd.notna(x):
        temp_pub = ["PMID:" + i.strip() for i in x.split(";")]
    else:
        temp_pub = None    
    processed_publications.append(temp_pub)

In [30]:
pd.Series(processed_publications).isna().sum()

processed_publications[0:3]

np.int64(80)

[['PMID:18423520', 'PMID:25574057', 'PMID:29140751'],
 ['PMID:21240275', 'PMID:21240277'],
 ['PMID:10502833',
  'PMID:1463019',
  'PMID:1598908',
  'PMID:2365823',
  'PMID:7649538']]

### source record url

UI really wants resource website urls like this. May need to adjust over time as website changes

In [36]:
## in parser, adjusted for single-row-level transform

"https://www.ebi.ac.uk/gene2phenotype/lgd/" +  df["g2p id"]

0       https://www.ebi.ac.uk/gene2phenotype/lgd/G2P00001
1       https://www.ebi.ac.uk/gene2phenotype/lgd/G2P00002
2       https://www.ebi.ac.uk/gene2phenotype/lgd/G2P00003
3       https://www.ebi.ac.uk/gene2phenotype/lgd/G2P00004
4       https://www.ebi.ac.uk/gene2phenotype/lgd/G2P00005
                              ...                        
3057    https://www.ebi.ac.uk/gene2phenotype/lgd/G2P00939
3058    https://www.ebi.ac.uk/gene2phenotype/lgd/G2P01393
3059    https://www.ebi.ac.uk/gene2phenotype/lgd/G2P01854
3060    https://www.ebi.ac.uk/gene2phenotype/lgd/G2P01857
3061    https://www.ebi.ac.uk/gene2phenotype/lgd/G2P01850
Name: g2p id, Length: 3061, dtype: object

### date (last reviewed)

In [34]:
## in parser, adjusted for single-row-level transform

[i[0:10] for i in df["date of last review"]][0:10]

df["date of last review"][0:10]

['2019-09-26',
 '2025-01-28',
 '2015-07-22',
 '2025-01-27',
 '2024-08-20',
 '2025-01-21',
 '2017-09-01',
 '2017-05-12',
 '2025-01-21',
 '2015-07-22']

0    2019-09-26 16:23:46+00:00
1    2025-01-28 23:09:54+00:00
2    2015-07-22 16:14:07+00:00
3    2025-01-27 14:24:27+00:00
4    2024-08-20 14:13:58+00:00
5    2025-01-21 22:30:30+00:00
6    2017-09-01 16:19:16+00:00
7    2017-05-12 10:48:00+00:00
8    2025-01-21 22:09:55+00:00
9    2015-07-22 16:14:08+00:00
Name: date of last review, dtype: object

### ID prefixes

In [37]:
## in parser, adjusted for single-row-level transform

df["hgnc id"] = [f"HGNC:{i}" for i in df["hgnc id"]]

df["hgnc id"][0:5]

0     HGNC:5017
1    HGNC:23845
2      HGNC:663
3      HGNC:882
4     HGNC:3583
Name: hgnc id, dtype: object

In [39]:
## in parser, adjusted for single-row-level transform

df["disease mim"].isna().sum()

df["disease mim"] = [f"OMIM:{i}" if pd.notna(i) else None for i in df["disease mim"]]

df["disease mim"].isna().sum()
df["disease mim"][0:5]

np.int64(338)

np.int64(338)

0    OMIM:612109
1    OMIM:613951
2    OMIM:207800
3    OMIM:210600
4    OMIM:300514
Name: disease mim, dtype: object

## Comparing to parser output

In [128]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3061 entries, 0 to 3061
Data columns (total 24 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   g2p id                              3061 non-null   object 
 1   gene symbol                         3061 non-null   object 
 2   gene mim                            3061 non-null   object 
 3   hgnc id                             3061 non-null   object 
 4   previous gene symbols               2831 non-null   object 
 5   disease name                        3061 non-null   object 
 6   disease mim                         2723 non-null   object 
 7   disease MONDO                       2891 non-null   object 
 8   allelic requirement                 3061 non-null   object 
 9   cross cutting modifier              395 non-null    object 
 10  confidence                          3061 non-null   object 
 11  variant consequence                 3049 non-nul

In [103]:
df[df["disease mim"].isna() & df["disease MONDO"].notna()].shape[0]

338

In [100]:
df["molecular mechanism"].value_counts()

molecular mechanism
loss of function                     2012
undetermined                          769
gain of function                      175
dominant negative                      98
undetermined non-loss-of-function       7
Name: count, dtype: int64

In [101]:
df["allelic requirement"].value_counts()

allelic requirement
biallelic_autosomal           1668
monoallelic_autosomal         1183
monoallelic_X_hemizygous       151
monoallelic_X_heterozygous      50
mitochondrial                    6
monoallelic_X                    2
monoallelic_Y_hemizygous         1
Name: count, dtype: int64

In [102]:
df["confidence"].value_counts()

confidence
definitive    1974
strong         840
moderate       247
Name: count, dtype: int64

## OLD: Pre-NodeNorming

Querying NodeNorm: send unique values (no duplicates!) from entire column in large batches -> generate mapping dict to use. 
<br>
__Not querying 1-by-1 or 1 row at a time: much slower__ and would involve sending duplicate IDs (unless saved dict is kept outside loop and checked) 

Not going to use NameResolver: not optimistic this would work anyways. My manual process of getting "better" disease IDs is to use the gene IDs, find the diseases they're linked to in OMIM and Monarch, and seeing if those match the data's disease name / phenotypes / publications. This is more complicated than just using NameResolver.

<div class="alert alert-block alert-danger">

Set the NodeNorm URL you want to use. 

In [41]:
## PARTIAL INCLUDE IN PARSER (only what's used)


## from BioThings annotator code: for interoperability between diff Python versions
try:
    from itertools import batched  # new in Python 3.12
except ImportError:
    from itertools import islice

    def batched(iterable, n):
        # batched('ABCDEFG', 3) → ABC DEF G
        if n < 1:
            raise ValueError("n must be at least one")
        iterator = iter(iterable)
        while batch := tuple(islice(iterator, n)):
            yield batch

nodenorm_url = "https://nodenorm.ci.transltr.io/get_normalized_nodes"

### Exploring: Genes

**2025-07-31 data:** Every row has at least 1 gene ID (HGNC column has no missing values). So no rows will be removed because there's no gene IDs to use for the pre-NodeNorming. 

In [42]:
df[["gene symbol", "hgnc id", "gene mim"]].info()

<class 'pandas.core.frame.DataFrame'>
Index: 3061 entries, 0 to 3061
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   gene symbol  3061 non-null   object
 1   hgnc id      3061 non-null   object
 2   gene mim     3061 non-null   object
dtypes: object(3)
memory usage: 95.7+ KB


#### HGNC

__Running Gene HGNC IDs through NodeNorm__


Catching potential mapping failures for later stats report

In [44]:
## get set of unique CURIEs to put into NodeNorm
hgnc = df["hgnc id"].dropna().unique()
len(hgnc)

2590

In [45]:
hgnc_nodenorm_mapping = {}

## set up variables to catch potential mapping failures
stats_hgnc_mapping_failures = {
    "unexpected_error": {},
    "nodenorm_returned_none": [],
    "wrong_category": {},
    "no_label": []
    
}

In [46]:
## larger batches are quicker
for batch in batched(hgnc, 1000):
    ## returns tuples -> cast to list
    req_body = {
        "curies": list(batch),
        "conflate": True,
    }
    r = requests.post(nodenorm_url, json=req_body)
    response = r.json()
    
    ## not doing dict comprehension. allows easier review, logic writing
    for k,v in response.items():
        ## catch unexpected errors
        try:
            ## if NodeNorm didn't have info on this ID, v will be None
            if v is not None:
                ## don't keep mapping if category is not the expected one
                if v["type"][0] == "biolink:Gene":
                    ## also throw out mapping if no primary label found
                    if v["id"].get("label"):
                        temp = {
                            k: {"primary_id": v["id"]["identifier"],
                                "primary_label": v["id"]["label"]
                               }
                        }
                        hgnc_nodenorm_mapping.update(temp)
                    else:
                        stats_hgnc_mapping_failures["no_label"].append(k)
#                         print(f"{k}: NodeNorm didn't find primary label. Not keeping this mapping.")
                else:
                    stats_hgnc_mapping_failures["wrong_category"].update({k: v["type"][0]})
#                     print(f'{k}: NodeNorm found different category {v["type"][0]}. Not keeping this mapping.')
            else:
                stats_hgnc_mapping_failures["nodenorm_returned_none"].append(k)
#                 print(f"{k}: NodeNorm didn't recognize this ID")
        except:
            stats_hgnc_mapping_failures["unexpected_error"].update({k: v})
            print(f'Encountered an unexpected error.')
            print(f'NodeNorm response key: {k}')
            print(f'NodeNorm response value: {v}')

In [47]:
len(hgnc_nodenorm_mapping)

stats_hgnc_mapping_failures

2590

{'unexpected_error': {},
 'nodenorm_returned_none': [],
 'wrong_category': {},
 'no_label': []}

#### OMIM

__Running Gene OMIM IDs through NodeNorm__

Catching potential mapping failures for later stats report. 

Pasted, adjusted from HGNC code blocks above.

In [52]:
df["gene mim"] = [f"OMIM:{i}" for i in df["gene mim"]]

df["gene mim"][0:5]

0    OMIM:142992
1    OMIM:613278
2    OMIM:608313
3    OMIM:601215
4    OMIM:300515
Name: gene mim, dtype: object

In [53]:
## get set of unique CURIEs to put into NodeNorm
gene_omim = df["gene mim"].dropna().unique()
len(gene_omim)

2590

In [54]:
gene_omim_nodenorm_mapping = {}

## set up variables to catch potential mapping failures
stats_gene_omim_mapping_failures = {
    "unexpected_error": {},
    "nodenorm_returned_none": [],
    "wrong_category": {},
    "no_label": []
    
}

In [55]:
## larger batches are quicker
for batch in batched(gene_omim, 1000):
    ## returns tuples -> cast to list
    req_body = {
        "curies": list(batch),
        "conflate": True,
    }
    r = requests.post(nodenorm_url, json=req_body)
    response = r.json()
    
    ## not doing dict comprehension. allows easier review, logic writing
    for k,v in response.items():
        ## catch unexpected errors
        try:
            ## if NodeNorm didn't have info on this ID, v will be None
            if v is not None:
                ## don't keep mapping if category is not the expected one
                if v["type"][0] == "biolink:Gene":
                    ## also throw out mapping if no primary label found
                    if v["id"].get("label"):
                        temp = {
                            k: {"primary_id": v["id"]["identifier"],
                                "primary_label": v["id"]["label"]
                               }
                        }
                        gene_omim_nodenorm_mapping.update(temp)
                    else:
                        stats_gene_omim_mapping_failures["no_label"].append(k)
#                         print(f"{k}: NodeNorm didn't find primary label. Not keeping this mapping.")
                else:
                    stats_gene_omim_mapping_failures["wrong_category"].update({k: v["type"][0]})
#                     print(f'{k}: NodeNorm found different category {v["type"][0]}. Not keeping this mapping.')
            else:
                stats_gene_omim_mapping_failures["nodenorm_returned_none"].append(k)
#                 print(f"{k}: NodeNorm didn't recognize this ID")
        except:
            stats_gene_omim_mapping_failures["unexpected_error"].update({k: v})
            print(f'Encountered an unexpected error.')
            print(f'NodeNorm response key: {k}')
            print(f'NodeNorm response value: {v}')

In [56]:
len(gene_omim_nodenorm_mapping)

stats_gene_omim_mapping_failures

2587

{'unexpected_error': {},
 'nodenorm_returned_none': [],
 'wrong_category': {'OMIM:109270': 'biolink:Disease',
  'OMIM:603273': 'biolink:Disease',
  'OMIM:604712': 'biolink:Disease'},
 'no_label': []}

#### Comparing HGNC vs OMIM

itertuples needs keys without spaces, so using a copy with adjusted column names

In [60]:
df_underscores = df.copy()

df_underscores.columns = df_underscores.columns.str.replace(" ", "_")

In [61]:
df_underscores.columns

Index(['g2p_id', 'gene_symbol', 'gene_mim', 'hgnc_id', 'previous_gene_symbols',
       'disease_name', 'disease_mim', 'disease_MONDO', 'allelic_requirement',
       'cross_cutting_modifier', 'confidence', 'variant_consequence',
       'variant_types', 'molecular_mechanism', 'molecular_mechanism_support',
       'molecular_mechanism_categorisation', 'molecular_mechanism_evidence',
       'phenotypes', 'publications', 'additional_mined_publications', 'panel',
       'comments', 'date_of_last_review', 'review'],
      dtype='object')

In [62]:
## if row has both IDs, look for diff in mappings from each ID
for row in df_underscores[["gene_mim", "hgnc_id"]].itertuples(index=False):
    ## has both IDs
    if pd.notna(row.gene_mim) and pd.notna(row.hgnc_id):
        ## if have NodeNorm mappings for both
        if gene_omim_nodenorm_mapping.get(row.gene_mim) and \
        hgnc_nodenorm_mapping.get(row.hgnc_id):
            ## check if mappings are diff
            if gene_omim_nodenorm_mapping[row.gene_mim]["primary_id"] != \
            hgnc_nodenorm_mapping[row.hgnc_id]["primary_id"]:
                print(row)

In [63]:
## look for differences in name between NodeNormed and original data

for row in df_underscores[["gene_symbol", "hgnc_id"]].itertuples(index=False):
    ## works because both columns have no missing values and there's no failed mappings
    ## if this changes, need to adjust this code block
    if row.gene_symbol != hgnc_nodenorm_mapping[row.hgnc_id]["primary_label"]:
        print(f"G2P name {row.gene_symbol}, ID {row.hgnc_id}")
        print(f'NodeNorm name {hgnc_nodenorm_mapping[row.hgnc_id]["primary_label"]}, ID {hgnc_nodenorm_mapping[row.hgnc_id]["primary_id"]}')
        print("\n")

**2025-12-10 data:** no more mismatched names


**2025-08-07 data:** 

Review of mismatched names:
* NodeNorm is correct on the official names of:
  * CENPJ -> should be CPAP
  * CCDC103 -> DNAAF19
  * CCDC115 -> VMA22
  * TMEM199 -> VMA12
  * RNU2-2P -> RNU2-2
* The rest look like mitochondrial genes, and NCBIGene main name seems to match G2P name, not NodeNorm -> messaged NodeNorm

#### Conclusions

<div class="alert alert-block alert-success">

**2025-12-10 data:** 
    
__Exploration__

* mapping failures with OMIM IDs
* when rows have both OMIM and HGNC IDs, there were no differences in NodeNorm mapping ("mismatches")
    
__Decision: Use HGNC ID column to generate NodeNorm values__

* less missing values (none right now), no mapping failures
* these IDs are probably only genes (vs OMIM ID namespace has multiple kinds of entities)

### Exploring: Diseases

There are many more missing IDs for Disease, compared to Gene. 

As mentioned at the beginning of the "Pre-NodeNorming" section, I won't be using NameResolver right now. 

__This means all rows w/o any disease IDs will be removed__ because they cannot be pre-NodeNormed. 

In [64]:
df[["disease name", "disease mim", "disease MONDO"]].info()

<class 'pandas.core.frame.DataFrame'>
Index: 3061 entries, 0 to 3061
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   disease name   3061 non-null   object
 1   disease mim    2723 non-null   object
 2   disease MONDO  2891 non-null   object
dtypes: object(3)
memory usage: 95.7+ KB


#### OMIM/orphanet

__Running OMIM/orphanet IDs through NodeNorm__

Catching mapping failures for later stats report

Pasted, adjusted from HGNC code blocks above.

In [65]:
## get set of unique CURIEs to put into NodeNorm
disease_OmOr = df["disease mim"].dropna().unique()
len(disease_OmOr)

2650

In [66]:
OmOr_nodenorm_mapping = {}

## set up variables to catch mapping failures
stats_OmOr_mapping_failures = {
    "unexpected_error": {},
    "nodenorm_returned_none": [],
    "wrong_category": {},
    "no_label": []
    
}

In [67]:
## larger batches are quicker
for batch in batched(disease_OmOr, 1000):
    ## returns tuples -> cast to list
    req_body = {
        "curies": list(batch),
        "conflate": True,
    }
    r = requests.post(nodenorm_url, json=req_body)
    response = r.json()
    
    ## not doing dict comprehension. allows easier review, logic writing
    for k,v in response.items():
        ## catch unexpected errors
        try:
            ## if NodeNorm didn't have info on this ID, v will be None
            if v is not None:
                ## don't keep mapping if category is not the expected one
                if v["type"][0] == "biolink:Disease":
                    ## also throw out mapping if no primary label found
                    if v["id"].get("label"):
                        temp = {
                            k: {"primary_id": v["id"]["identifier"],
                                "primary_label": v["id"]["label"]
                               }
                        }
                        OmOr_nodenorm_mapping.update(temp)
                    else:
                        stats_OmOr_mapping_failures["no_label"].append(k)
#                         print(f"{k}: NodeNorm didn't find primary label. Not keeping this mapping.")
                else:
                    stats_OmOr_mapping_failures["wrong_category"].update({k: v["type"][0]})
#                     print(f'{k}: NodeNorm found different category {v["type"][0]}. Not keeping this mapping.')
            else:
                stats_OmOr_mapping_failures["nodenorm_returned_none"].append(k)
#                 print(f"{k}: NodeNorm didn't recognize this ID")
        except:
            stats_OmOr_mapping_failures["unexpected_error"].update({k: v})
            print(f'Encountered an unexpected error.')
            print(f'NodeNorm response key: {k}')
            print(f'NodeNorm response value: {v}')

In [69]:
## calculate stats: number of rows affected by each type of mapping failure
stats_OmOr_mapping_failures.update({
    "n_rows_none": df[df["disease mim"].isin(stats_OmOr_mapping_failures["nodenorm_returned_none"])].shape[0],
    "n_rows_wrong_category": df[df["disease mim"].isin(stats_OmOr_mapping_failures["wrong_category"].keys())].shape[0],
    "n_rows_no_label": df[df["disease mim"].isin(stats_OmOr_mapping_failures["no_label"])].shape[0]
})

In [70]:
len(OmOr_nodenorm_mapping)

stats_OmOr_mapping_failures["unexpected_error"]

len(stats_OmOr_mapping_failures["nodenorm_returned_none"])
len(stats_OmOr_mapping_failures["wrong_category"])
len(stats_OmOr_mapping_failures["no_label"])

2646

{}

4

0

0

In [71]:
## code used to review mapping failures 

stats_OmOr_mapping_failures["nodenorm_returned_none"]

stats_OmOr_mapping_failures["wrong_category"]

stats_OmOr_mapping_failures["no_label"]

['OMIM:601884', 'OMIM:133701', 'OMIM:133700', 'OMIM:150800']

{}

[]

In [73]:
## code used to review mapping failures 

df[df["disease mim"] == "OMIM:133701"]

,g2p id,gene symbol,gene mim,hgnc id,previous gene symbols,disease name,disease mim,disease MONDO,allelic requirement,cross cutting modifier,confidence,variant consequence,variant types,molecular mechanism,molecular mechanism support,molecular mechanism categorisation,molecular mechanism evidence,phenotypes,publications,additional mined publications,panel,comments,date of last review,review
425,G2P00486,EXT2,OMIM:608210,HGNC:3513,SOTV,EXT2-related multiple exostoses,OMIM:133701,MONDO:0007586,monoallelic_autosomal,NaN,definitive,absent gene product,NaN,loss of function,inferred,NaN,NaN,HP:0000006; HP:0000896; HP:0000918; HP:0002318...,9326317,10429361; 10671060; 10679937; 10738008; 111700...,DD; Cancer; Skeletal,NaN,2023-05-24 09:07:28+00:00,NaN


<div class="alert alert-block alert-info">

**Update 2025-12-10:**
    
4 cases where NodeNorm returned None: 
* OMIM:601884 - [valid ID](https://omim.org/entry/601884), but it doesn't seem to be a disease (previously reviewed, reported to EBI gene2pheno, NodeNorm)
* OMIM:133701 - [valid disease ID](https://omim.org/entry/133701), NodeNorm issue
* OMIM:133700 - [valid disease ID](https://omim.org/entry/133700), NodeNorm issue (previously reviewed, reported)
* OMIM:150800 - [valid disease ID](https://omim.org/entry/150800), NodeNorm issue

   
1 case where NodeNorm category was something else (currently, always Gene): 
* OMIM:188400 - [valid disease ID](https://omim.org/entry/188400), NodeNorm error (previously reviewed, reported)

<div class="alert alert-block alert-success">
    
**2025-08-07 data:** 

MONDO mappings are correct when OMIM mapping failed. 
    
But it doesn't cover all rows that have failures.

In [74]:
## code used to check how many rows have OMIM failure + MONDO ID 

df[df["disease mim"].isin(stats_OmOr_mapping_failures["nodenorm_returned_none"]) & 
   df["disease MONDO"].notna()]

df[df["disease mim"].isin(stats_OmOr_mapping_failures["wrong_category"].keys()) & 
   df["disease MONDO"].notna()]

df[df["disease mim"].isin(stats_OmOr_mapping_failures["no_label"]) & 
   df["disease MONDO"].notna()]

,g2p id,gene symbol,gene mim,hgnc id,previous gene symbols,disease name,disease mim,disease MONDO,allelic requirement,cross cutting modifier,confidence,variant consequence,variant types,molecular mechanism,molecular mechanism support,molecular mechanism categorisation,molecular mechanism evidence,phenotypes,publications,additional mined publications,panel,comments,date of last review,review
425,G2P00486,EXT2,OMIM:608210,HGNC:3513,SOTV,EXT2-related multiple exostoses,OMIM:133701,MONDO:0007586,monoallelic_autosomal,NaN,definitive,absent gene product,NaN,loss of function,inferred,NaN,NaN,HP:0000006; HP:0000896; HP:0000918; HP:0002318...,9326317,10429361; 10671060; 10679937; 10738008; 111700...,DD; Cancer; Skeletal,NaN,2023-05-24 09:07:28+00:00,NaN
634,G2P00733,EXT1,OMIM:608177,HGNC:3512,LGCR; LGS; TTV,EXT1-related multiple exostoses,OMIM:133700,MONDO:0007585,monoallelic_autosomal,NaN,definitive,absent gene product,NaN,loss of function,inferred,NaN,NaN,HP:0000006; HP:0000076; HP:0000252; HP:0000365...,15253765; 7550340; 8981950; 9326317,10429361; 10679937; 11170095; 11180615; 114329...,DD; Cancer; Skeletal,NaN,2023-05-24 09:07:28+00:00,NaN
2721,G2P01806,FH,OMIM:136850,HGNC:3700,NaN,FH-related leiomyomatosis and renal cell cancer,OMIM:150800,MONDO:0007888,monoallelic_autosomal,NaN,definitive,absent gene product,NaN,loss of function,inferred,NaN,NaN,HP:0000119; HP:0001574; HP:0001939; HP:0002664,11865300; 12772087; 15663510,NaN,Skin; Cancer,NaN,2017-09-01 16:19:15+00:00,NaN


,g2p id,gene symbol,gene mim,hgnc id,previous gene symbols,disease name,disease mim,disease MONDO,allelic requirement,cross cutting modifier,confidence,variant consequence,variant types,molecular mechanism,molecular mechanism support,molecular mechanism categorisation,molecular mechanism evidence,phenotypes,publications,additional mined publications,panel,comments,date of last review,review


,g2p id,gene symbol,gene mim,hgnc id,previous gene symbols,disease name,disease mim,disease MONDO,allelic requirement,cross cutting modifier,confidence,variant consequence,variant types,molecular mechanism,molecular mechanism support,molecular mechanism categorisation,molecular mechanism evidence,phenotypes,publications,additional mined publications,panel,comments,date of last review,review


#### MONDO

__Running MONDO IDs through NodeNorm__

Catching potential mapping failures for later stats report

Pasted, adjusted from Disease OMIM/orphanet code blocks above.

In [75]:
## get set of unique CURIEs to put into NodeNorm
mondo = df["disease MONDO"].dropna().unique()
len(mondo)

2670

In [76]:
mondo_nodenorm_mapping = {}

## set up variables to catch mapping failures
stats_mondo_mapping_failures = {
    "unexpected_error": {},
    "nodenorm_returned_none": [],
    "wrong_category": {},
    "no_label": []
    
}

In [77]:
## larger batches are quicker
for batch in batched(mondo, 1000):
    ## returns tuples -> cast to list
    req_body = {
        "curies": list(batch),
        "conflate": True,
    }
    r = requests.post(nodenorm_url, json=req_body)
    response = r.json()
    
    ## not doing dict comprehension. allows easier review, logic writing
    for k,v in response.items():
        ## catch unexpected errors
        try:
            ## if NodeNorm didn't have info on this ID, v will be None
            if v is not None:
                ## don't keep mapping if category is not the expected one
                if v["type"][0] == "biolink:Disease":
                    ## also throw out mapping if no primary label found
                    if v["id"].get("label"):
                        temp = {
                            k: {"primary_id": v["id"]["identifier"],
                                "primary_label": v["id"]["label"]
                               }
                        }
                        mondo_nodenorm_mapping.update(temp)
                    else:
                        stats_mondo_mapping_failures["no_label"].append(k)
#                         print(f"{k}: NodeNorm didn't find primary label. Not keeping this mapping.")
                else:
                    stats_mondo_mapping_failures["wrong_category"].update({k: v["type"][0]})
#                     print(f'{k}: NodeNorm found different category {v["type"][0]}. Not keeping this mapping.')
            else:
                stats_mondo_mapping_failures["nodenorm_returned_none"].append(k)
#                 print(f"{k}: NodeNorm didn't recognize this ID")
        except:
            stats_mondo_mapping_failures["unexpected_error"].update({k: v})
            print(f'Encountered an unexpected error.')
            print(f'NodeNorm response key: {k}')
            print(f'NodeNorm response value: {v}')

In [78]:

## calculate stats: number of rows affected by each type of mapping failure
stats_mondo_mapping_failures.update({
    "n_rows_none": df[df["disease MONDO"].isin(stats_mondo_mapping_failures["nodenorm_returned_none"])].shape[0],
    "n_rows_wrong_category": df[df["disease MONDO"].isin(stats_mondo_mapping_failures["wrong_category"].keys())].shape[0],
    "n_rows_no_label": df[df["disease MONDO"].isin(stats_mondo_mapping_failures["no_label"])].shape[0]
})

In [79]:
len(mondo_nodenorm_mapping)

stats_mondo_mapping_failures

2669

{'unexpected_error': {},
 'nodenorm_returned_none': ['MONDO:0700360'],
 'wrong_category': {},
 'no_label': [],
 'n_rows_none': 1,
 'n_rows_wrong_category': 0,
 'n_rows_no_label': 0}

#### Comparing OMIM/orphanet vs MONDO

In [81]:
## using notebook-specific df_underscores

## if row has both IDs, look for diff in mappings from each ID

## list of tuples (omim/orpha, mondo)
mismatches = []

for row in df_underscores[["disease_mim", "disease_MONDO"]].itertuples(index=False):
    ## has both IDs
    if pd.notna(row.disease_mim) and pd.notna(row.disease_MONDO):
        ## if have NodeNorm mappings for both
        if OmOr_nodenorm_mapping.get(row.disease_mim) and \
        mondo_nodenorm_mapping.get(row.disease_MONDO):
            ## check if mappings are diff
            if OmOr_nodenorm_mapping[row.disease_mim]["primary_id"] != \
            mondo_nodenorm_mapping[row.disease_MONDO]["primary_id"]:
                mismatches.append((row.disease_mim, row.disease_MONDO))

print(f"There's {len(mismatches)} mismatches between the OMIM/orphanet and MONDO NodeNorm mappings.")

There's 76 mismatches between the OMIM/orphanet and MONDO NodeNorm mappings.


In [82]:
## code chunk to review mismatches 1 by 1
mismatches[12]

('OMIM:614114', 'MONDO:0013582')

In [83]:
## code chunk to review mismatches 1 by 1

OmOr_nodenorm_mapping["OMIM:614114"]
mondo_nodenorm_mapping["MONDO:0013582"]

{'primary_id': 'OMIM:614114',
 'primary_label': 'Mosaic Variegated Aneuploidy Syndrome 2'}

{'primary_id': 'MONDO:0013582',
 'primary_label': 'mosaic variegated aneuploidy syndrome 2'}

In [85]:
## code chunk to review mismatches 1 by 1

df[df["disease mim"] == "OMIM:614114"]
df[df["disease MONDO"] == "MONDO:0013582"]

,g2p id,gene symbol,gene mim,hgnc id,previous gene symbols,disease name,disease mim,disease MONDO,allelic requirement,cross cutting modifier,confidence,variant consequence,variant types,molecular mechanism,molecular mechanism support,molecular mechanism categorisation,molecular mechanism evidence,phenotypes,publications,additional mined publications,panel,comments,date of last review,review
857,G2P00991,CEP57,OMIM:607951,HGNC:30794,KIAA0092; TRANSLOKIN; TSP57,CEP57-related mosaic variegated aneuploidy syn...,OMIM:614114,MONDO:0013582,biallelic_autosomal,NaN,definitive,absent gene product,NaN,loss of function,inferred,NaN,NaN,HP:0000007; HP:0000252; HP:0000268; HP:0000276...,12116237; 21552266,24259107; 30010053; 30147898; 31943948; 328618...,DD,NaN,2015-07-22 16:15:04+00:00,NaN


,g2p id,gene symbol,gene mim,hgnc id,previous gene symbols,disease name,disease mim,disease MONDO,allelic requirement,cross cutting modifier,confidence,variant consequence,variant types,molecular mechanism,molecular mechanism support,molecular mechanism categorisation,molecular mechanism evidence,phenotypes,publications,additional mined publications,panel,comments,date of last review,review
857,G2P00991,CEP57,OMIM:607951,HGNC:30794,KIAA0092; TRANSLOKIN; TSP57,CEP57-related mosaic variegated aneuploidy syn...,OMIM:614114,MONDO:0013582,biallelic_autosomal,NaN,definitive,absent gene product,NaN,loss of function,inferred,NaN,NaN,HP:0000007; HP:0000252; HP:0000268; HP:0000276...,12116237; 21552266,24259107; 30010053; 30147898; 31943948; 328618...,DD,NaN,2015-07-22 16:15:04+00:00,NaN


<div class="alert alert-block alert-info">    

**2025-08-07 data:** 

__Review of OMIM vs MONDO NodeNorm mismatches__ (45 total, reviewed 23)

None were orphanet.    
**Looked over the pairs that were reviewed last time** (2025-02-28) and more

---

__10: OMIM's mapping is better__

> __2: Mondo ID is related but wrong__ -> emailed EBI gene2pheno w/ example
> * 'OMIM:101000', 'MONDO:0008075': omim is correct type of schwannomatosis (NF2/type 2), vs mondo is a sibling (both under neurofibromatosis MONDO:0021061)
>   * NodeNorm should map omim to MONDO:0007039 but isn't -> messaged NodeNorm
> * 'OMIM:613987', __'MONDO:0009136'__: omim is correct recessive 2, but mondo is recessive 1 (diff gene? Confusing because Monarch page links to gene NHP2 but OMIM page doesn't)
>   * NodeNorm should map omim to MONDO:0013519 but isn't -> messaged NodeNorm  

> __8: Mondo ID is too general__ (can see on Monarch website) -> emailed EBI gene2pheno w/ example
> * 'OMIM:300696', 'MONDO:0010680': omim maps to MONDO:0010401, child of the mondo
> * 'OMIM:304120', 'MONDO:0019027': omim maps to MONDO:0010571 (syndrome type 2), child of the mondo (syndrome)
> * 'OMIM:610019', 'MONDO:0005129': omim maps to MONDO:0012395 (cataract 18), child of the mondo (cataract)
> * 'OMIM:611726', 'MONDO:0016295': omim maps to MONDO:0012721, child of the mondo 
> * 'OMIM:602668', 'MONDO:0016107': omim maps to MONDO:0011266 (type 2), child of the mondo
> * 'OMIM:203200', 'MONDO:0018910': omim maps to MONDO:0008746 (type 2), child of the mondo
> * 'OMIM:614328', 'MONDO:0017411': omim maps to MONDO:0013693 (type 1), child of the mondo
> * 'OMIM:175800', 'MONDO:0006602': omim maps to MONDO:0008290 (1, mibelli type), grandchild of the mondo



**1: MONDO's mapping is better**
* 'OMIM:613723', 'MONDO:0009181': mondo matches the disease name and phenotypes listed in the record better than the omim 


**2: Unsure**
* 'OMIM:614886', 'MONDO:0100270': both are technically correct. OMIM maps to more specific MONDO:0013951 (only child of the mondo provided) 
* 'OMIM:612463', 'MONDO:0019992': the records involved (G2P00591, G2P02117) are for gene GNAS. This gene is tied to both pseudopseudohypoparathyroidism AND specific subtypes of pseudohypoparathyroidism. Both records seem to have somewhat incorrect ID assignments. 


**10: NodeNorm error**
* 'OMIM:224230', 'MONDO:0009136': both are recessive 1, NodeNorm should map to same entity  -> messaged NodeNorm
* 'OMIM:613988', 'MONDO:0013520': both are autosomal recessive 3, NodeNorm should map to same entity -> messaged NodeNorm
* 'OMIM:616353', 'MONDO:0009136': both are recessive 6, NodeNorm should map to same entity -> messaged NodeNorm
* 'OMIM:257300', 'MONDO:0009759': both are same entity according to Monarch 
* 'OMIM:609322', 'MONDO:0012252': both are same entity according to Monarch 
* 'OMIM:162200', 'MONDO:0018975': both are same entity according to Monarch 
* 'OMIM:162300', 'MONDO:0008082': both are same entity according to Monarch
* 'OMIM:240500', 'MONDO:0009413': both are same entity according to Monarch
* 'OMIM:614114', 'MONDO:0013582': both are same entity according to Monarch
* 'OMIM:610738', 'MONDO:0012548': both are same entity according to Monarch

<div class="alert alert-block alert-success">    

**CONCLUSION**
    
**2025-08-07 data:** 

Prefer the disease_mim column (OMIM/orphanet IDs) to MONDO, if a row has both

#### Checking MONDO data

Above, I decided the OMIM/orphanet disease IDs were better. 

However, I wondered if the MONDO IDs were accurate to the disease name when there weren't OMIM/orphanet IDs. Then they could be used for NodeNorming and less data would be dropped because there weren't IDs for NodeNorming.

In [86]:
## get the data that has MONDO, doesn't have OMIM/orphanet

df_mondo_only = df[df["disease mim"].isna() & df["disease MONDO"].notna()].copy()

mondo_only = df_mondo_only["disease MONDO"].dropna().unique()

In [87]:
## saving stats on data with only MONDO ID

stats_mondo_only = {
    "n_rows": df_mondo_only.shape[0],
    "n_names": len(mondo_only)
}

stats_mondo_only["n_rows"]
stats_mondo_only["n_names"]

338

201

In [93]:
## code chunk used to review some of the data

pd.set_option('display.max_colwidth', None)
pd.reset_option('display.max_colwidth')

# df_mondo_only[df_mondo_only["disease MONDO"] == mondo_only[4]]
# df[df["disease MONDO"] == "MONDO:0010912"]

# df[df["disease name"] == "DSC2-related arrhythmogenic right ventricular cardiomyopathy"]
# df[df["gene symbol"] == "ALPL"]

# df_mondo_only[df_mondo_only["panel"].str.contains("Ear")]

<div class="alert alert-block alert-info">    

**2025-08-07 data:** 

__Reviewed some of the data__

Method: look at rows (aka MONDO ID/disease name pairs). **Looked over the pairs that were reviewed last time** (2025-02-28) and more. Some weren't in the data (perhaps filtered out earlier, removed from dataset, or now had omim + mondo IDs in row).

    
__Summary__
* 30 rows / 296 (~10%)
* None were wrong! 
* Could tell EBI gene2pheno of issues but they are similar to those listed in mismatch mapping section

__Details__

__2 MONDO is too general__ 
* "MONDO:0018965" (Alport syndrome) for "COL4A5-related Alport syndrome". The COL4A5-specific version is a child term: MONDO:0010520/OMIM:301050  (X-linked)
* "MONDO:0014980" (cone-rod dystrophy and hearing loss) for "CEP78-related cone-rod dystrophy and hearing loss". The CEP78-specific version is a child term: MONDO:0020778/OMIM:617236


__2 Unsure -> TELL EBI GENE2PHENO?__
* "MONDO:0005129" for "CYP51A1-related congenital cataract": mondo is cataract, which is not wrong but kinda general. MONDO:0033853 seems better (correlated with gene, matches phenotypes, orphanet ref uses one of the ref papers) 
* "MONDO:0020367" for "MYOC-related juvenile open angle glaucoma": while mondo is almost-exact name match, it's not directly linked to gene. Instead, its child disease is directly linked to gene MONDO:0007664/OMIM:137750 (glaucoma 1, open angle, A) 


__11 Okay (using general term is fine)__
* **{counts as two}** "MONDO:0024676" (childhood kidney Wilms tumor) for "CTR9-related Wilms tumour", "TRIM28-": couldn't find better mapping. TRIM28 is correlated to parent term (kidney Wilms tumor). 
* "MONDO:0019118" for "CLN3-related retinal dystrophy": couldn't find better mapping 
* "MONDO:0018570" for "ALPL-related hypophosphatasia": there are more specific terms, but it's unclear which one to use here. So using more general one is fine
* "MONDO:0015469" for "PRRX1-related craniosynostosis": couldn't find better mapping 
* "MONDO:0018094" for "SNAI2-related Waardenburg syndrome": while MONDO maps to the more specific type 2 (MONDO:0019517), this specific type's inheritance pattern doesn't match (dominant/monoallelic vs recessive/biallelic)
* "MONDO:0100124" for "NAA10-related nonpecific severe intellectual disability": there are more specific terms, but it's unclear which one to use here. So using more general one is fine
* **{counts as two}** "MONDO:0001071" for "SCN2A-related nonspecific severe intellectual disability", "AFF3-related intellectual disability": there are more specific terms linked to the gene, but it's unclear which one to use here. So using more general one is fine
* "MONDO:0100062" for "FLNA-related epileptic encephalopathy": there are more specific terms linked to the gene, but it's unclear which one to use here. So using more general one is fine
* "MONDO:0700092" for "PRRT2-related intellectual developmental disorder": there are more specific terms linked to the gene, but it's unclear which one to use here. So using more general one is fine


__15 Great__
* "MONDO:0012506" for "DSC2-related arrhythmogenic right ventricular cardiomyopathy"
* "MONDO:0011001" for "SCN5A-related Brugada syndrome"
* "MONDO:0013262" for "MYH7-related dilated cardiomyopathy"
* "MONDO:0013369" for "TNNI3-related hypertrophic cardiomyopathy"
* "MONDO:0010946" for "PRKAG2-related cardiomyopathy"
* "MONDO:0014143" for "RIT1-related Noonan syndrome"
* "MONDO:0014214" for "DYNC2I1-related short-rib polydactyly"
* "MONDO:0013522" for "TINF2-related dyskeratosis congenita"
* "MONDO:0010215" for "ERCC4-related xeroderma pigmentosum, group F"
* "MONDO:0009735" for "SPINK5-related Netherton syndrome"
* "MONDO:0007566" for "TGFBR1-related multiple self-healing squamous epithelioma"
* "MONDO:0007485" for "TERC-related dyskeratosis congenita"
* "MONDO:0032835" for "MIR140-related spondyloepiphyseal dysplasia, Nishimura type"
* "MONDO:0001071" for "GSPT2-related intellectual disability"
* "MONDO:0009869" for "SOX9-related Pierre Robin sequence"

#### Conclusions

<div class="alert alert-block alert-success">

**2025-08-07 data:** 
    
__Exploration__

* some rows have no disease IDs -> will filter these out (can't NodeNorm)
* very few NodeNorm mapping failures for IDs
* when rows have both OMIM and MONDO ID, use OMIM ID - generally more accurate
  * while it would be nice to use MONDO ID if NodeNorm fails for OMIM ID, the common ingest pipeline won't have this level of sophistication to do this
* when row only has MONDO ID, can use it - accurate enough now!
    
__Decision: Use OMIM ID column as primary, MONDO ID if don't have OMIM ID__

### Stats on rows removed during NodeNorming

This section prints the statistics on rows in the original data that were removed. 

(Uses variables generated during the previous section "Pre-NodeNorming")

<div class="alert alert-block alert-success">

**2025-08-07 data:** 

Genes: No rows removed due to lack of IDs for NodeNorming or NodeNorm mapping issues.

In [95]:
## PARTIAL INCLUDE IN PARSER

print("Gene Pre-NodeNorming\n")

## HGNC NodeNorm issues: none, but showing anyways
print("HGNC NodeNorm mapping failures:")

print(f'IDs with no data in NodeNorm: {len(stats_hgnc_mapping_failures["nodenorm_returned_none"])}')
print(f'IDs with the wrong NodeNormed category: {len(stats_hgnc_mapping_failures["wrong_category"])}')
print(f'IDs with no label in NodeNorm: {len(stats_hgnc_mapping_failures["no_label"])}')

Gene Pre-NodeNorming

HGNC NodeNorm mapping failures:
IDs with no data in NodeNorm: 0
IDs with the wrong NodeNormed category: 0
IDs with no label in NodeNorm: 0


<div class="alert alert-block alert-success">

**2025-08-07 data:** 
    
__Diseases: some rows removed__ due to lack of IDs for NodeNorming. A few rows removed due NodeNorm mapping issues.

In [96]:
stats_OmOr_mapping_failures
stats_mondo_mapping_failures

{'unexpected_error': {},
 'nodenorm_returned_none': ['OMIM:601884',
  'OMIM:133701',
  'OMIM:133700',
  'OMIM:150800'],
 'wrong_category': {},
 'no_label': [],
 'n_rows_none': 4,
 'n_rows_wrong_category': 0,
 'n_rows_no_label': 0}

{'unexpected_error': {},
 'nodenorm_returned_none': ['MONDO:0700360'],
 'wrong_category': {},
 'no_label': [],
 'n_rows_none': 1,
 'n_rows_wrong_category': 0,
 'n_rows_no_label': 0}

In [98]:
## PARTIAL INCLUDE IN PARSER

print("Disease Pre-NodeNorming\n")

## OMIM NodeNorm issues
print("OMIM NodeNorm mapping failures:")

print(f'{stats_OmOr_mapping_failures["n_rows_none"]} row(s) for '
      f'{len(stats_OmOr_mapping_failures["nodenorm_returned_none"])} '
      f'IDs with no data in NodeNorm')

print(f'{stats_OmOr_mapping_failures["n_rows_wrong_category"]} row(s) for '
      f'{len(stats_OmOr_mapping_failures["wrong_category"])} '
      f'IDs with the wrong NodeNormed category')

print(f'{stats_OmOr_mapping_failures["n_rows_no_label"]} row(s) for '
      f'{len(stats_OmOr_mapping_failures["no_label"])} '
      f'IDs with no label in NodeNorm')

## MONDO NodeNorm issues
print("\n")
print("MONDO NodeNorm mapping failures:")

print(f'{stats_mondo_mapping_failures["n_rows_none"]} row(s) for '
      f'{len(stats_mondo_mapping_failures["nodenorm_returned_none"])} '
      f'IDs with no data in NodeNorm')

print(f'{stats_mondo_mapping_failures["n_rows_wrong_category"]} row(s) for '
      f'{len(stats_mondo_mapping_failures["wrong_category"])} '
      f'IDs with the wrong NodeNormed category')

print(f'{stats_mondo_mapping_failures["n_rows_no_label"]} row(s) for '
      f'{len(stats_mondo_mapping_failures["no_label"])} '
      f'IDs with no label in NodeNorm')

Disease Pre-NodeNorming

OMIM NodeNorm mapping failures:
4 row(s) for 4 IDs with no data in NodeNorm
0 row(s) for 0 IDs with the wrong NodeNormed category
0 row(s) for 0 IDs with no label in NodeNorm


MONDO NodeNorm mapping failures:
1 row(s) for 1 IDs with no data in NodeNorm
0 row(s) for 0 IDs with the wrong NodeNormed category
0 row(s) for 0 IDs with no label in NodeNorm


### Adding NodeNorm data, removing rows

Using gene HGNC and disease OMIM/orphanet or MONDO IDs for pre-NodeNorming

In [104]:
df_underscores.shape

(3061, 24)

In [105]:
## create single column to use for disease NodeNorming
df_underscores["disease_original_id"] = [
    x.disease_mim if pd.notna(x.disease_mim)
    else x.disease_MONDO if pd.notna(x.disease_MONDO)
    else pd.NA
    for x in df_underscores[["disease_mim", "disease_MONDO"]].itertuples()
]

In [117]:
## double-check new column logic

df_underscores[df_underscores["disease_mim"].isna() & df_underscores["disease_MONDO"].notna()].shape[0]

df_underscores[df_underscores["disease_original_id"].str.contains("MONDO", na=False)].shape[0]

338

338

In [118]:
df_underscores.dropna(subset="disease_original_id", inplace=True, ignore_index=True)

In [119]:

## Gene: assumes no missing values
df_underscores["gene_nodenorm_id"] = [hgnc_nodenorm_mapping[i]["primary_id"] for i in df_underscores["hgnc_id"]]
df_underscores["gene_nodenorm_label"] = [hgnc_nodenorm_mapping[i]["primary_label"] for i in df_underscores["hgnc_id"]]

df_underscores["disease_nodenorm_id"] = [
    ## should work for OMIM and orphanet
    OmOr_nodenorm_mapping[i]["primary_id"] if OmOr_nodenorm_mapping.get(i)
    else mondo_nodenorm_mapping[i]["primary_id"] if mondo_nodenorm_mapping.get(i)
    else pd.NA
    for i in df_underscores["disease_original_id"]
]

df_underscores["disease_nodenorm_label"] = [
    ## should work for OMIM and orphanet
    OmOr_nodenorm_mapping[i]["primary_label"] if OmOr_nodenorm_mapping.get(i)
    else mondo_nodenorm_mapping[i]["primary_label"] if mondo_nodenorm_mapping.get(i)
    else pd.NA
    for i in df_underscores["disease_original_id"]
]

In [120]:
## move disease_original_id to end for easier comparison

df_underscores["disease_original_id"] = df_underscores.pop("disease_original_id")

In [121]:
df_underscores.head()

,g2p_id,gene_symbol,gene_mim,hgnc_id,previous_gene_symbols,disease_name,disease_mim,disease_MONDO,allelic_requirement,cross_cutting_modifier,confidence,variant_consequence,variant_types,molecular_mechanism,molecular_mechanism_support,molecular_mechanism_categorisation,molecular_mechanism_evidence,phenotypes,publications,additional_mined_publications,panel,comments,date_of_last_review,review,gene_nodenorm_id,gene_nodenorm_label,disease_nodenorm_id,disease_nodenorm_label,disease_original_id
0,G2P00001,HMX1,OMIM:142992,HGNC:5017,H6; NKX5-3,HMX1-related oculoauricular syndrome,OMIM:612109,MONDO:0012802,biallelic_autosomal,NaN,definitive,absent gene product,NaN,loss of function,inferred,NaN,NaN,HP:0000007; HP:0000356; HP:0000482; HP:0000505...,18423520; 25574057; 29140751,21417677; 31469246; 35946463,DD; Eye,NaN,2019-09-26 16:23:46+00:00,NaN,NCBIGene:3166,HMX1,MONDO:0012802,oculoauricular syndrome,OMIM:612109
1,G2P00002,SLX4,OMIM:613278,HGNC:23845,BTBD12; FANCP; KIAA1784; KIAA1987,SLX4-related Fanconi anemia,OMIM:613951,MONDO:0013499,biallelic_autosomal,NaN,definitive,absent gene product,NaN,loss of function,inferred,NaN,NaN,HP:0000007; HP:0000028; HP:0000085; HP:0000125...,21240275; 21240277,21476996; 23033263; 26841305; 30047418; 347541...,DD,NaN,2025-01-28 23:09:54+00:00,NaN,NCBIGene:84464,SLX4,MONDO:0013499,Fanconi anemia complementation group P,OMIM:613951
2,G2P00003,ARG1,OMIM:608313,HGNC:663,NaN,ARG1-related argininemia,OMIM:207800,MONDO:0008814,biallelic_autosomal,NaN,definitive,absent gene product,NaN,loss of function,inferred,NaN,NaN,HP:0000007; HP:0000737; HP:0000752; HP:0001249...,10502833; 1463019; 1598908; 2365823; 7649538,15565656; 19052914; 19562505; 19936428; 213103...,DD,NaN,2015-07-22 16:14:07+00:00,NaN,NCBIGene:383,ARG1,MONDO:0008814,Argininemia,OMIM:207800
3,G2P00004,ATR,OMIM:601215,HGNC:882,FRP1; MEC1; SCKL; SCKL1,ATR-related Seckel syndrome,OMIM:210600,MONDO:0008869,biallelic_autosomal,NaN,strong,absent gene product,NaN,loss of function,inferred,NaN,NaN,HP:0000007; HP:0000028; HP:0000047; HP:0000175...,10889046; 12640452; 23111928; 30199583,10631133; 15279811; 21669506; 23144622; 272354...,DD; Skeletal,NaN,2025-01-27 14:24:27+00:00,NaN,NCBIGene:545,ATR,MONDO:0008869,Seckel syndrome 1,OMIM:210600
4,G2P00005,FANCB,OMIM:300515,HGNC:3583,FAAP95; FAB; FLJ34064,FANCB-related Fanconi anemia,OMIM:300514,MONDO:0010351,monoallelic_X_hemizygous,NaN,definitive,absent gene product,NaN,loss of function,inferred,NaN,NaN,HP:0000083; HP:0000100; HP:0000119; HP:0000707...,15502827; 16679491; 21910217; 22052692; 236135...,24416387; 31351673,DD; Skin,NaN,2024-08-20 14:13:58+00:00,NaN,NCBIGene:2187,FANCB,MONDO:0010351,Fanconi anemia complementation group B,OMIM:300514


In [122]:
df_only_nodenormed = df_underscores.dropna(subset=["gene_nodenorm_id", "gene_nodenorm_label", 
                                       "disease_nodenorm_id", "disease_nodenorm_label", 
                                       "disease_original_id"],
                              ignore_index=True).copy()

In [124]:
## same! so it works as expected

df_only_nodenormed.shape

(3056, 29)

In [125]:
df_only_nodenormed.info(memory_usage="deep")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3056 entries, 0 to 3055
Data columns (total 29 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   g2p_id                              3056 non-null   object 
 1   gene_symbol                         3056 non-null   object 
 2   gene_mim                            3056 non-null   object 
 3   hgnc_id                             3056 non-null   object 
 4   previous_gene_symbols               2827 non-null   object 
 5   disease_name                        3056 non-null   object 
 6   disease_mim                         2719 non-null   object 
 7   disease_MONDO                       2887 non-null   object 
 8   allelic_requirement                 3056 non-null   object 
 9   cross_cutting_modifier              393 non-null    object 
 10  confidence                          3056 non-null   object 
 11  variant_consequence                 3044 no

## OLD: Generating documents

In [76]:
## want jsonlines format

import jsonlines

### BioThings-type parser 

Includes `_id` set to g2p_id value

In [ ]:
# ## code chunk for testing parts of inner code

# for row in df_only_nodenormeddf.itertuples(index=False):
#     document = {
#         "_id": row.g2p_id,
#         "subject": row.gene_nodenorm_id,
#         "sources": [
#             {
#                 "resource_id": "infores:ebi-gene2phenotype",
#                 "resource_role": "primary_knowledge_source",
#                 "source_record_urls": [row.g2p_record_url]
#             }
#         ],
#         "attributes": [
#             {
#                 "attribute_type_id": "biolink:original_subject",
#                 "value": row.hgnc_id
#             }
#         ]
#     }
#     if pd.notna(row.publications):
#         document["attributes"].append(
#             {
#                 "attribute_type_id": "biolink:publications",
#                 "value": ["PMID:" + i.strip() for i in row.publications.split(";")]
#             }
#         )
#     document
#     break

In [ ]:
# ## put into parser (format): DONE
# ##   don't save in array, yield each document instead

# ## GENERATING DOCS, saving in array
# documents = []

# ## using itertuples because it's faster, preserves datatypes
# for row in df_only_nodenormed.itertuples(index=False):
#     ## simple assignments: no NA or "if"
#     document = {
#         "_id": row.g2p_id,
#         "subject": row.gene_nodenorm_id,
#         "qualifiers": [  ## needs data-modeling/TRAPI validation review
#             {
#                 "qualifier_type_id": "biolink:subject_form_or_variant_qualifier",
#                 "qualifier_value": "genetic_variant_form"
#             }
#         ],
#         "object": row.disease_nodenorm_id,
#         "sources": [
#             {
#                 "resource_id": "infores:ebi-gene2phenotype",
#                 "resource_role": "primary_knowledge_source",
#                 "source_record_urls": [row.g2p_record_url]
#             }
#         ],
#         "attributes": [
#             {
#                 "attribute_type_id": "biolink:knowledge_level",
#                 "value": "knowledge_assertion"
#             },
#             {
#                 "attribute_type_id": "biolink:agent_type",
#                 "value": "manual_agent"
#             },
#             {
#                 "attribute_type_id": "biolink:original_subject",
#                 "original_attribute_name": "hgnc id",  ## original column name
#                 "value": row.hgnc_id
#             },
#             {   ## currently, after NodeNorming, no NAs in OMIM/orphanet column
#                 "attribute_type_id": "biolink:original_object",
#                 "original_attribute_name": "disease mim",  ## original column name
#                 "value": row.disease_mim
#             },
#             {   ## needs data-modeling/TRAPI validation review
#                 ## EBI gene2pheno website calls this "Last Updated"/"Last Updated On"
#                 "attribute_type_id": "biolink:update_date",
#                 "original_attribute_name": "date of last review",  ## original column name
#                 "value": str(row.date_of_last_review)
#             },
#         ]
#     }
    
#     ## more complex assignments ("if", handling NA). When value is NA, list comprehension with split won't work
#     ## predicate
#     if row.confidence == "limited":
#         document["predicate"] = "biolink:related_to"
#     elif row.confidence in ["moderate", "strong", "definitive"]:
#         document["predicate"] = "biolink:causes"
#     else:
#         raise ValueError(f"Unexpected confidence value during predicate mapping: {row.confidence}. Adjust parser.")
#     ## publications
#     if pd.notna(row.publications):
#         document["attributes"].append(
#             {
#                 "attribute_type_id": "biolink:publications",
#                 "value": ["PMID:" + i.strip() for i in row.publications.split(";")]
#             }
#         )
    
#     documents.append(document)

### File: List of TRAPI edges

This code isn't in parser.py

Doesn't have `_id`! Doesn't include original_attribute_name for original_subject, original_object, update_date.

In [77]:
df_only_nodenormed.columns

Index(['g2p_id', 'gene_symbol', 'gene_mim', 'hgnc_id', 'previous_gene_symbols',
       'disease_name', 'disease_mim', 'disease_MONDO', 'allelic_requirement',
       'cross_cutting_modifier', 'confidence', 'variant_consequence',
       'variant_types', 'molecular_mechanism', 'molecular_mechanism_support',
       'molecular_mechanism_categorisation', 'molecular_mechanism_evidence',
       'phenotypes', 'publications', 'panel', 'comments',
       'date_of_last_review', 'review', 'g2p_record_url', 'gene_nodenorm_id',
       'gene_nodenorm_label', 'disease_nodenorm_id', 'disease_nodenorm_label',
       'disease_original_id'],
      dtype='object')

In [78]:
## uncomment to write file

# ## wrapped with file writer, otherwise contents very similar to before
# ## commented out original_attribute_name

# with jsonlines.open('EBIgene2pheno_trapi_edges.jsonl', mode='w', compact=True) as trapi_writer:

#     ## using itertuples because it's faster, preserves datatypes
#     for row in df_only_nodenormed.itertuples(index=False):
        
#         ## simple assignments: no NA or "if"
#         document = {
#             "subject": row.gene_nodenorm_id,
#             "predicate": "biolink:associated_with",
#             "qualifiers": [
#                 {
#                     "qualifier_type_id": "biolink:qualified_predicate",
#                     "qualifier_value": "biolink:causes"
#                 },
#                 {
#                     "qualifier_type_id": "biolink:subject_form_or_variant_qualifier",
#                     "qualifier_value": molecularmech_mappings[row.molecular_mechanism]
#                 },
#             ],
#             "object": row.disease_nodenorm_id,
#             "sources": [
#                 {
#                     "resource_id": "infores:ebi-gene2phenotype",
#                     "resource_role": "primary_knowledge_source",
#                     "source_record_urls": [row.g2p_record_url]
#                 }
#             ],
#             "attributes": [
#                 {
#                     "attribute_type_id": "biolink:knowledge_level",
#                     "value": "knowledge_assertion"
#                 },
#                 {
#                     "attribute_type_id": "biolink:agent_type",
#                     "value": "manual_agent"
#                 },
#                 {
#                     "attribute_type_id": "biolink:original_subject",
# #                     "original_attribute_name": "hgnc id",  ## original column name
#                     "value": row.hgnc_id
#                 },
#                 {   ## uses column created during parsing
#                     "attribute_type_id": "biolink:original_object",
#                     "value": row.disease_original_id
#                 },
#                 ## 2025-09-04: removing right now because common ingest pipeline can't handle it properly
# #                 {   ## EBI gene2pheno website calls this "Last Updated"/"Last Updated On"
# #                     "attribute_type_id": "biolink:update_date",
# # #                     "original_attribute_name": "date of last review",  ## original column name
# #                     "value": str(row.date_of_last_review)
# #                 },
#                 {
#                     "attribute_type_id": "biolink:allelic_requirement",
#                     "value": allelicreq_mappings[row.allelic_requirement]
#                 },
#             ]
#         }

#         ## more complex assignments ("if", handling NA). When value is NA, list comprehension with split won't work
#         ## publications
#         if pd.notna(row.publications):
#             document["attributes"].append(
#                 {
#                     "attribute_type_id": "biolink:publications",
#                     "value": ["PMID:" + i.strip() for i in row.publications.split(";")]
#                 }
#             )
            
#         ## doing so it doesn't print
#         bytes = trapi_writer.write(document)

### File: KGX edges

This code isn't in parser.py

Doesn't include original_attribute_name for original_subject, original_object, update_date

In [79]:
## uncomment to write file

# with jsonlines.open('EBIgene2pheno_kgx_edges.jsonl', mode='w', compact=True) as kgx_edges_writer:

#     ## using itertuples because it's faster, preserves datatypes
#     for row in df_only_nodenormed.itertuples(index=False):
        
#         ## simple assignments: no NA or "if"
#         document = {
#             "subject": row.gene_nodenorm_id,
#             "predicate": "biolink:associated_with",
#             "qualified_predicate": "biolink:causes",
#             "subject_form_or_variant_qualifier": molecularmech_mappings[row.molecular_mechanism],
#             "object": row.disease_nodenorm_id,
#             "sources": [
#                 {
#                     "resource_id": "infores:ebi-gene2phenotype",
#                     "resource_role": "primary_knowledge_source",
#                     "source_record_urls": [row.g2p_record_url]
#                 }
#             ],
#             "knowledge_level": "knowledge_assertion",
#             "agent_type": "manual_agent",
#             "original_subject": row.hgnc_id,
#             ## uses column created during parsing
#             "original_object": row.disease_original_id,
#             ## 2025-09-04: removing right now because common ingest pipeline can't handle it properly
# #             ## EBI gene2pheno website calls this "Last Updated"/"Last Updated On"
# #             "update_date": str(row.date_of_last_review),
#             "allelic_requirement": allelicreq_mappings[row.allelic_requirement],
#         }

#         ## more complex assignments ("if", handling NA). When value is NA, list comprehension with split won't work
#         ## publications
#         if pd.notna(row.publications):
#             document["publications"] = ["PMID:" + i.strip() for i in row.publications.split(";")]

            
#         ## doing so it doesn't print
#         bytes = kgx_edges_writer.write(document)

### File: KGX nodes

This code isn't in parser.py

Requires id and category. name and other properties (basically node attributes) are optional. 

In [126]:
nodenormed_genes_final = df_only_nodenormed[["gene_nodenorm_id", "gene_nodenorm_label"]].drop_duplicates()
nodenormed_diseases_final = df_only_nodenormed[["disease_nodenorm_id", "disease_nodenorm_label"]].drop_duplicates()

nodenormed_genes_final
nodenormed_diseases_final

,gene_nodenorm_id,gene_nodenorm_label
0,NCBIGene:3166,HMX1
1,NCBIGene:84464,SLX4
2,NCBIGene:383,ARG1
3,NCBIGene:545,ATR
4,NCBIGene:2187,FANCB
...,...,...
3047,NCBIGene:374462,PTPRQ
3048,NCBIGene:5269,SERPINB6
3049,NCBIGene:7007,TECTA
3051,NCBIGene:286262,TPRN


,disease_nodenorm_id,disease_nodenorm_label
0,MONDO:0012802,oculoauricular syndrome
1,MONDO:0013499,Fanconi anemia complementation group P
2,MONDO:0008814,Argininemia
3,MONDO:0008869,Seckel syndrome 1
4,MONDO:0010351,Fanconi anemia complementation group B
...,...,...
3051,MONDO:0013215,autosomal recessive nonsyndromic hearing loss 79
3052,MONDO:0013963,autosomal recessive nonsyndromic hearing loss 93
3053,MONDO:0007395,craniofacial-deafness-hand syndrome
3054,MONDO:0011519,autosomal dominant nonsyndromic hearing loss 23


In [81]:
## uncomment to write file

# with jsonlines.open('EBIgene2pheno_kgx_nodes.jsonl', mode='w', compact=True) as kgx_nodes_writer:
    
#     ## using itertuples because it's faster, preserves datatypes
#     for row in nodenormed_genes_final.itertuples(index=False):
#         ## doing so it doesn't print
#         bytes = kgx_nodes_writer.write({
#             "id": row.gene_nodenorm_id,
#             "name": row.gene_nodenorm_label,            
#             ## hard-coded because during pre-NodeNorm process, only kept entities with this primary category
#             "category": ["biolink:Gene"]
            
#         })

#     ## using itertuples because it's faster, preserves datatypes
#     for row in nodenormed_diseases_final.itertuples(index=False):
#         ## doing so it doesn't print
#         bytes = kgx_nodes_writer.write({
#             "id": row.disease_nodenorm_id,
#             "name": row.disease_nodenorm_label,
#             ## hard-coded because during pre-NodeNorm process, only kept entities with this primary category
#             "category": ["biolink:Disease"]
#         })

In [127]:
nodenormed_genes_final.shape[0] + nodenormed_diseases_final.shape[0]

5419

## DEFUNCT: Checking documents

In [ ]:
# len(documents)

In [ ]:
# df_only_nodenormed.info()

In [ ]:
## code chunk for finding rows -> then look up corresponding doc by idx
# df_only_nodenormed[df_only_nodenormed["disease_mim"].str.contains("orphanet", na=False)]
# df_only_nodenormed[df_only_nodenormed["confidence"] == "limited"]
# df_only_nodenormed[df_only_nodenormed["publications"].isna()]
# df_only_nodenormed[~df_only_nodenormed["publications"].str.contains(";", na=True)]



# df_only_nodenormed[df_only_nodenormed["previous_gene_symbols"].isna()]
# df_only_nodenormed[df_only_nodenormed["disease_MONDO"].notna()]

In [ ]:
# pprint(documents[34])

# documents[416]